# Data Preparation - Study 2a

## Loading the libraries

In [1]:
import numpy as np
import pandas as pd
import json
import re
import yfinance as yf

## Reading the LLM responses

In [2]:
df = pd.read_csv('./data/ageRisk_complete.csv', index_col=0)

## Extracting the financial advice

In [3]:
def create_long(df):
    # Create a new DataFrame to store the long format
    long_df = pd.DataFrame(columns=["ai", "age", "risk-taking", "cond", "iteration", "investment_type", "name", "ticker_symbol", "amount_to_invest"])

    # Iterate over each row in the original DataFrame
    for _, row in df.iterrows():
        ai = row['ai']
        age = row['age']
        risk_taking = row['risk-taking']
        cond = row['cond']
        iteration = row['iteration']
        investments_new = row['investments']

        # Check if 'investments_new' is empty
        if not investments_new:
            # Create a new row with NAs
            new_row = {
                'ai': ai,
                'age': age,
                'risk-taking': risk_taking,
                'cond': cond,
                'iteration': iteration,
                'investment_type': None,
                'name': None,
                'ticker_symbol': None,
                'amount_to_invest': None
            }

            # Append the new row to the long format DataFrame
            long_df.loc[len(long_df)] = new_row
        else:
            # Iterate over each investment in the 'investments_new' list
            investments_new = investments_new.replace("'", "\"")
            investments_new = json.loads(investments_new)
            for investment in investments_new:
                investment_type = investment['Investment Type']
                name = investment['Name']
                ticker_symbol = investment['Ticker Symbol']
                amount_to_invest = investment['Amount to Invest']

                # Create a new row with the extracted information
                new_row = {
                    'ai': ai,
                    'age': age,
                    'risk-taking': risk_taking,
                    'cond': cond,
                    'iteration': iteration,
                    'investment_type': investment_type,
                    'name': name,
                    'ticker_symbol': ticker_symbol,
                    'amount_to_invest': amount_to_invest
                }

                # Append the new row to the long format DataFrame
                long_df.loc[len(long_df)] = new_row
    return long_df

def remove_invalid_rows(df):
    # we conducted a visual inspection of the data and found the following parsing problems  due to unexpected new lines/ summary rows.
    print(f"before shape: {df.shape}")
    conditions = []
    
    for condition in conditions:
        # Delete the rows that satisfy the conditions
        df = df[~condition]
    
    print(f"after shape: {df.shape}")
    return df

def fix_ticker(df):
    df.loc[df['name'] == 'Bitcoin (BTC)', 'ticker_symbol'] = 'BTC-USD'
    df.loc[df['ticker_symbol'] == 'BTCUSD', 'ticker_symbol'] = 'BTC-USD'
    df.loc[df['ticker_symbol'] == 'FB', 'ticker_symbol'] = 'META'
    df.loc[df['ticker_symbol'] == 'BTC - Coinbase', 'ticker_symbol'] = 'BTC-USD'
    df.loc[df['ticker_symbol'] == 'ETH - Coinbase', 'ticker_symbol'] = 'ETH-USD'
    df.loc[df['ticker_symbol'] == 'BTCUSD', 'ticker_symbol'] = 'BTC-USD'
    df.loc[df['ticker_symbol'] == 'XRP', 'ticker_symbol'] = 'XRP-USD'
    df.loc[df['ticker_symbol'] == 'DOGE', 'ticker_symbol'] = 'DOGE-USD'
    df.loc[df['ticker_symbol'] == 'DOT', 'ticker_symbol'] = 'DOT-USD'
    df.loc[df['ticker_symbol'] == 'FB', 'ticker_symbol'] = 'META'
    return df

def extract_ticker(value):
    if isinstance(value, str):
        ticker = value.split(' (', 1)[0]
        return ticker
    return value

def clean_ticker(df):
    df = fix_ticker(df)
    nas = ['', 'n/a', 'N/A', '-', '--', '---', 'nan']
    df['ticker_symbol'] = df['ticker_symbol'].str.replace('**','')
    df['ticker_symbol'] = df['ticker_symbol'].str.replace('`', '')
    df['ticker_symbol'] = df['ticker_symbol'].replace(nas, None)
    df['ticker_symbol'] = df['ticker_symbol'].apply(extract_ticker)
    df['ticker_symbol'] = df['ticker_symbol'].replace('BTC', 'BTC-USD').replace('ETH', 'ETH-USD').replace('BNB', 'BNB-USD').replace('ADA', 'ADA-USD')#BNB, XRP
    df['ticker_symbol'] = df['ticker_symbol'].replace('ALLY', None).replace('GSBANK', None).replace('(Private)', None)
    df['ticker_symbol'] = df['ticker_symbol'].replace('FTCH', 'FTCHF')
    return df

def clean_amount(value):
    if '%' in value:
        percentage_match = re.search(r'(\d+)%', value)
        if percentage_match:
            percentage = int(percentage_match.group(1))
            amount = int(percentage / 100 * 10000) # Take $10,000 as a basis
            return amount
    else:
        amount_match = re.search(r'\$([\d,]+)', value)
        if amount_match:
            amount_str = amount_match.group(1).replace(',', '')  # Remove commas from the amount string
            amount = int(amount_str)
            return amount
    return None

def clean_data(df):
    df = remove_invalid_rows(df)
    df = clean_ticker(df) # clean all tickers
    df['amount_to_invest'] = df['amount_to_invest'].apply(clean_amount)
    df = df[df['amount_to_invest'] != 10000] # remove summary rows
    df = df[df[['amount_to_invest', 'ticker_symbol']].notna().any(axis=1)] # remove additional parsing errors
    return df

asset_dict = {}

def get_asset_info(ticker):
    if ticker == None:
        return pd.Series([None, None, None])
    if ticker in asset_dict:
        return asset_dict[ticker]
    try:
        # get asset info
        asset = yf.Ticker(ticker)
        info = asset.info
        assetType = info.get('quoteType')
        country = None
        sector = None
        
        if assetType.lower() == 'equity':
            country = info.get('country')
            sector = info.get('sector')
        ret = pd.Series([assetType, country, sector])
        asset_dict[ticker]  = ret
        return ret
    except:
        print(f"didn't find ticker: '{ticker}'")
        asset_dict[ticker] = pd.Series([None, None, None])
        return pd.Series([None, None, None])


def complete_asset_type(row):
    if row['asset_type'] == None or pd.isnull(row['asset_type']):
        investmentType = str(row['investment_type']).lower()
        name = str(row['name']).lower()

        if 'bond' in investmentType or 'bond' in name or 'treasury securities' in investmentType:
            return 'BOND'
        elif 'saving' in investmentType or 'saving' in name or 'custodial' in investmentType:
            return 'SAVING'
        elif 'money' in investmentType or 'money' in name or 'cash' in investmentType :
            return 'MONEYMARKET'
        elif 'certificate' in investmentType or 'cd' in investmentType or 'certificate' in name or 'deposit' in investmentType:
            return 'DEPOSIT'
        elif 'estate' in investmentType or 'estate' in name:
            return 'REALESTATE'
        elif 'ira' in investmentType or 'ira' in name:
            return 'IRA'
        elif 'annuit' in investmentType or 'annuit' in name:
            return 'ANNUITY'
        elif 'venture' in investmentType or 'venture' in name or 'startup' in investmentType:
            return 'VENTURECAPITAL'
        elif 'seed' in investmentType or 'seed' in name:
            return 'SEEDINVEST'
        elif 'private' in investmentType or 'private' in name:
            return 'PRIVATEEQUITY'
        elif 'fund' in investmentType or 'fund' in name:
            return 'MUTUALFUND'
        elif 'auto' in investmentType:
            return 'ROBO'
        elif 'lending' in investmentType:
            return 'P2PLENDING'
            
    return row['asset_type']



In [4]:
df = create_long(df)
df = clean_data(df)
df = df[~df['amount_to_invest'].isna()] # still one summary row which needs to be excluded
df[['asset_type', 'country', 'sector']] = df['ticker_symbol'].apply(get_asset_info)
df['asset_type'] = df.apply(complete_asset_type, axis=1)
df.to_csv('./data/ageRisk_long.csv')

before shape: (999, 9)
after shape: (999, 9)
